In [2]:
import os
from dotenv import load_dotenv
from agents import Agent, function_tool, Runner, trace, ModelSettings, AsyncOpenAI, OpenAIChatCompletionsModel, WebSearchTool
import smtplib
import asyncio
from email.message import EmailMessage


In [2]:
instrucitons1 = """
You are a email writer that helps draft professional sales emails to customers. Always stay professional and polite.
"""
instrucitons2 = """
You are email writer that drafts witty and humorous emails to draw the customers attention. Your tone is casual and fun yet polite.
"""

instructions3 = """
You are a email writer that acts a sales executive and drafts professional, concise and persuasive emails to customers. Your tone is polite, professional and persuasive.
"""

In [3]:
sales_agent1 = Agent(name="Sales Agent 1", instructions=instrucitons1, model="gpt-4o-mini")
sales_agent2 = Agent(name="Sales Agent 2", instructions=instrucitons2, model="gpt-4o-mini")
sales_agent3 = Agent(name="Sales Agent 3", instructions=instructions3, model="gpt-4o-mini")

In [7]:
message = "You need to draft a cold sales email to send to potential customers."
with trace("Sales Agents"):
    result = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for result in result]


In [7]:
for output in outputs:
    print(output + "\n\n")


Subject: Unlock New Opportunities with [Your Company Name]

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I am reaching out from [Your Company Name]. We specialize in [briefly describe your product/service] that helps businesses like yours [mention key benefits or solutions].

In today’s competitive landscape, maximizing efficiency and staying ahead of the curve is crucial. Our [product/service] has been designed to [explain a core benefit, e.g., save time, reduce costs, enhance productivity], and I believe it could add significant value to your operations.

I would love the opportunity to discuss how we can support [Recipient's Company Name] in achieving its goals. Would you be available for a brief call or meeting next week? Please let me know your availability, and I will do my best to accommodate.

Thank you for considering this opportunity. I look forward to the possibility of collaborating with you.

Best regards,

[Your Name]  
[Your J

In [8]:
instructions4 = """
You are an email picker, out of all the emails presented to you, you have to pick the best one. 
Consider yourself as a customer and pick the email that besr resonates with you. Return the email only do not add any other text or explanation.
"""

In [9]:
sales_picker_agent = Agent(name="Sales Picker Agent", instructions=instructions4, model="gpt-4o-mini")

In [10]:

message = "You need to pick the best email from the following options:\n\n" + "\n\n".join(outputs)

with trace("Sales Manager"):
    result = await Runner.run(sales_picker_agent, message)

print(result.final_output)

Subject: Unleash Your Inner Superhero with Our Services! 🦸‍♂️💥

Hey there [First Name],

Hope you’re having a fantastic day! 🌟 I promise to be quick—like that last slice of pizza that disappears at midnight. 🍕

I’m reaching out because we’ve got some super cool stuff happening at [Your Company Name], and I think it might just tickle your fancy. Picture this: a world where your daily tasks are as manageable as finding a cat video on the internet. Sounds good, right?

Here’s the scoop: we specialize in [briefly describe your services or products], and let’s be honest, everyone could do with a sprinkle of magic in their routine. 🪄✨

Now, I know what you’re thinking: “Another sales email? Yawn!” But hold your horses! I’m not just here to sell you stuff; I’m here to help solve problems, lighten workloads, and maybe even provide a few laughs along the way. 😂

Curious to learn more? Let’s hop on a quick call. Don’t worry, I promise I won’t ask about your favorite pizza topping… unless you wan

In [11]:
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")

In [5]:
def send_email(subject : str, text_body : str, html_body :str) -> str:
    """
    Sends an email with the given subject, text body, and HTML body.

    Args:
    subject (str): The subject of the email.
    text_body (str): The plain text content of the email.
    html_body (str): The HTML content of the email.
    """
    msg = EmailMessage()
    msg['from'] = os.getenv("EMAIL_ADDRESS")
    msg['to'] = os.getenv("EMAIL_ADDRESS")
    msg['subject'] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype='html')

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as smtp:
        smtp.starttls()
        smtp.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        smtp.send_message(msg)
    return "Email sent successfully!"

In [6]:
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

'Email sent successfully!'

In [12]:
@function_tool
def send_email(subject : str, text_body : str, html_body :str) -> str:
    """
    Sends an email with the given subject, text body, and HTML body.

    Args:
    subject (str): The subject of the email.
    text_body (str): The plain text content of the email.
    html_body (str): The HTML content of the email.
    """
    msg = EmailMessage()
    msg['from'] = os.getenv("EMAIL_ADDRESS")
    msg['to'] = os.getenv("EMAIL_ADDRESS")
    msg['subject'] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype='html')

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as smtp:
        smtp.starttls()
        smtp.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        smtp.send_message(msg)
    return "Email sent successfully!"

In [13]:
send_email.params_json_schema

{'properties': {'subject': {'title': 'Subject', 'type': 'string'},
  'text_body': {'title': 'Text Body', 'type': 'string'},
  'html_body': {'title': 'Html Body', 'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_args',
 'type': 'object',
 'additionalProperties': False}

In [14]:
message = """Imagine yourself as a customer and you have give 3 emails and " 
"you have to pick the best one in the persepctive of a customer and use the give tool to send and email."""

settings = ModelSettings(tool_choice="required")

manager_agent = Agent(name="Sales Manager Agent", instructions=message, model="gpt-4o-mini", tools=[send_email], model_settings=settings)

user_prompt = "You have to pick the best email from the following options:\n\n" + "\n\n".join(outputs) + "\n\nPlease use the send_email tool to send the selected email."

with trace("Sales Manager"):
    result = await Runner.run(manager_agent, user_prompt)

print(result.final_output)

The selected email, "Unlock New Opportunities for Your Business," has been sent successfully. If you need any further assistance, feel free to ask!


In [20]:
openai_api_key = os.getenv("OPENAI_API_KEY")
gemini_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

Step 1 Get openai compatible base urls for Gemini and Groq.

In [21]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

Step 2 Create python library client for each of the three LLM providers (OpenAI, Gemini, Groq) using the base urls and api keys.

In [22]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=gemini_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

Step 3 create the model objects for each of the three LLM providers using the clients created in step 2.

In [23]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
groq_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [30]:
instructions = """
You are a sales agent that drafts professional emails for company anyAI that help customers in creating tailored AI agents for their business requirements.
The tone of the email needs to be professional, with some humour and wit to draw the customers attention. The email should be concise, persuasive and polite.
Just give the email in the output do not add any other text or explanation.
"""

In [31]:
gemini_agent = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
groq_agent = Agent(name="Groq Sales Agent", instructions=instructions, model=groq_model)

In [32]:
description = "Use this tool to generate a professional sales email for anyAI that helps customers in creating tailored AI agents for their business requirements. The tone of the email needs to be professional, with some humour and wit to draw the customers attention. The email should be concise, persuasive and polite."

tool1 = gemini_agent.as_tool(tool_name="tool1", tool_description=description)
tool2 = groq_agent.as_tool(tool_name="tool2", tool_description=description)

In [33]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")


@function_tool
def send_email(subject:str, text_body:str, html_body:str) -> str:
    """
    Send an email with the given subject, text body and HTML body.

    Arguments:
    subject : The subject of the email.
    text_body : The content of the email in plain text format.
    html_body : The content of the email in HTML format.
    """
    msg = EmailMessage()
    msg["from"] = EMAIL_ADDRESS
    msg["to"] = EMAIL_ADDRESS
    msg["subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")
    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as smtp:
        smtp.starttls()
        smtp.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        smtp.send_message(msg)

    return "Email sent successfully!"

In [34]:
instructions = """"
You are a sales manager your task is to pick the best email using the tools provided by the sales agents. You have to pick the best email in the perspective of a customer and use the given tool to send an email.
"""

tools = [tool1, tool2, send_email]

sales_manager = Agent(name="Sales Manager Agent", instructions=instructions, model="gpt-5.4-mini", tools=tools)

In [35]:
task = "Pick an email and send it to the customer using the given tools."

with trace("Sales Manager"):
    result = await Runner.run(sales_manager, task)
result.final_output

'Sent successfully.'

In [41]:
instructions = """
You are a serach agent, give a term you search the web for it and return the summary of the search results. The summary should be concise and informative. 
Do not include any personal opinions or biases in the summary. It should less than 300 words.
"""

task = "What is the difference between RAG and MCP?"

In [42]:
tools = [WebSearchTool()]

settings = ModelSettings(tool_choice="required")

search_agent = Agent(name="Search Agent", instructions=instructions, model="gpt-4o-mini", tools=tools, model_settings=settings)

In [43]:
with trace("searching the web"):
    result = await Runner.run(search_agent, task)

print(result.final_output)

Retrieval-Augmented Generation (RAG) and Model Context Protocol (MCP) are two distinct approaches for enhancing the capabilities of large language models (LLMs), each addressing different aspects of AI functionality.

**Retrieval-Augmented Generation (RAG):**
RAG focuses on improving the factual accuracy of LLM outputs by integrating external knowledge. It retrieves relevant documents from a pre-indexed corpus and incorporates them into the model's prompt before generating a response. This method is particularly effective for tasks requiring semantic search across large, mostly static datasets, such as knowledge bases or documentation. RAG operates in a one-way data flow: retrieve → inject → generate, and is best suited for applications where the data is large and infrequently updated. ([intersystems.com](https://www.intersystems.com/resources/rag-vs-mcp-what-each-does-when-to-use-both/?utm_source=openai))

**Model Context Protocol (MCP):**
MCP, on the other hand, provides a standardiz

In [1]:
def WebSearchItem(BaseModel):
    reason: str = Field(description="The reason why the search term is relevant to the query.")
    query: str = Field(description="The query term to use for the web search.")


def WebSearchPlan(BaseModel):
    search_terms: list[WebSearchItem] = Field(description="A list of search terms to use for the web search.")

In [ ]:
instructions = """
You are a planner agent, given a query your task is to create a plan for web search. You have to come up with a list of search terms that are relevant to the query and provide a reason for each search term. The output should be in the form of a WebSearchPlan object.
"""



planner_agent = Agent(name="planner agent", instructions=instructions, model="gpt-5.4-mini", output_type=WebSearchPlan)

result = await Runner.run(planner_agent, "What is the difference between RAG and MCP?")
print(result.final_output)